In [30]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
!pip install better-profanity

In [5]:
# CELL 2: imports, device, seed
import random, re, torch
from datasets import load_dataset, Dataset

random.seed(0); torch.manual_seed(0)       # fixed seeds -> repeatable splits for your report
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
assert DEVICE == "cuda", "Switch the runtime to GPU (T4) first"

PREFIX = "Clean this sentence: "           # our own instruction; CoEdIT is used to 'instruction: text' inputs

Running on: cuda


In [6]:
# CELL 3: load ParaDetox, auto-detect the column names, split by TOXIC sentence (so no leakage into test)
raw = load_dataset("s-nlp/paradetox")
split = "train" if "train" in raw else list(raw.keys())[0]
pd_ds = raw[split]
print(pd_ds.column_names, len(pd_ds))
print(pd_ds[0])                             # <- paste this output to me if anything looks off

TOX = next(c for c in pd_ds.column_names if "toxic" in c.lower())
NEU = next(c for c in pd_ds.column_names if "neutral" in c.lower())
print("using columns:", TOX, "->", NEU)

# hold out 500 unique toxic sentences as the test set BEFORE building anything else
tox_unique = sorted({r for r in pd_ds[TOX] if r})
random.shuffle(tox_unique)
test_tox = set(tox_unique[:500])

real_train, real_test = [], []
for r in pd_ds:
    if not r[TOX] or not r[NEU]:
        continue
    pair = {"src": PREFIX + r[TOX], "tgt": r[NEU]}
    (real_test if r[TOX] in test_tox else real_train).append(pair)
print("real pairs  train:", len(real_train), " test:", len(real_test))

README.md:   0%|          | 0.00/7.59k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/2.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19744 [00:00<?, ? examples/s]

['en_toxic_comment', 'en_neutral_comment'] 19744
{'en_toxic_comment': 'he had steel balls too !', 'en_neutral_comment': 'he was brave too!'}
using columns: en_toxic_comment -> en_neutral_comment
real pairs  train: 18924  test: 820


In [24]:
# CELL 4: synthetic slang/typo pairs. Noisy version -> original clean sentence (meaning preserved by construction).
# Built from TRAIN neutral sentences only, so the test set stays untouched.
NOISE = {"going to": "gonna", "want to": "wanna", "have to": "gotta", "kind of": "kinda",
         "you": "u", "your": "ur", "please": "pls", "thanks": "thx", "honestly": "tbh",
         "I do not know": "idk", "because": "cuz", "though": "tho", "oh my god": "omg",
         "I dont know": "idk"}

def add_noise(s):
    for formal, slang in NOISE.items():
        if random.random() < 0.7:                                   # not always, so the data is varied
            s = re.sub(rf"\b{re.escape(formal)}\b", slang, s, flags=re.I)
    if random.random() < 0.5: s = s.replace("'", "")                # don't -> dont
    if random.random() < 0.5: s = s.lower()                         # lost capitalization
    if random.random() < 0.3 and len(s) > 6:                        # swap two adjacent letters = typo
        i = random.randrange(len(s) - 1)
        s = s[:i] + s[i + 1] + s[i] + s[i + 2:]
    return s

clean_pool = list({p["tgt"] for p in real_train})
synth = [{"src": PREFIX + add_noise(t), "tgt": t} for t in random.sample(clean_pool, min(4000, len(clean_pool)))]
for ex in synth[:5]:
    print(ex)                                                       # sanity check the direction: noisy -> clean

{'src': 'Clean this sentence: sometimes, i just sit here on twitter, thinking i am not stupid.', 'tgt': 'Sometimes, I just sit here on twitter, thinking I am not stupid.'}
{'src': 'Clean this sentence: World ucp 82 greenland!', 'tgt': 'World cup 82 greenland!'}
{'src': 'Clean this sentence: This people is now working on giving the largest tax cut t othe rich .', 'tgt': 'This people is now working on giving the largest tax cut to the rich .'}
{'src': 'Clean this sentence: nah , i just wanna say what ever i want and not get called out .', 'tgt': 'nah , i just wanna say what ever i want and not get called out .'}
{'src': "Clean this sentence: it's time have some bbq", 'tgt': "it's time have some bbq"}


In [26]:
# CELL 5: replay data = a slice of CoEdIT's OWN training data (apache-2.0). Keeps the original editing skills alive. To avoid forgetting its orignal use.
co = load_dataset("grammarly/coedit")["train"]
co_sample = co.shuffle(seed=0).select(range(3000))
replay = [{"src": r["src"], "tgt": r["tgt"]} for r in co_sample]    # src already looks like "instruction: text"
print(replay[0])

train = real_train + synth + replay
random.shuffle(train)
print("total training pairs:", len(train), "| real:", len(real_train), "synthetic:", len(synth), "replay:", len(replay))

README.md:   0%|          | 0.00/1.88k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 19.7MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/692k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/69071 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1712 [00:00<?, ? examples/s]

{'src': "Improve the grammaticality of this text: it's a pity", 'tgt': "It's a pity"}
total training pairs: 25924 | real: 18924 synthetic: 4000 replay: 3000


In [31]:
# CELL 6: base model + LoRA adapters
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BASE = "grammarly/coedit-large"
tok_ft = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE)      # fp32 on purpose: T5 + fp16 is unstable

# LoRA idea: freeze the original weights and train two tiny matrices (rank r) next to each attention weight.
# Output = W x + (alpha / r) * B A x   -> only A and B are trained, so ~1% of parameters and the base stays intact
lora = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q", "v"], bias="none")   # "q" and "v" = T5's attention query/value layers
model_ft = get_peft_model(base, lora)
model_ft.print_trainable_parameters()                   # sanity check: should be around 1% trainable

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 4,718,592 || all params: 787,811,328 || trainable%: 0.5989


In [34]:
# CELL 7: tokenize + train
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

def tokenize(batch):
    x = tok_ft(batch["src"], max_length=96, truncation=True)
    x["labels"] = tok_ft(text_target=batch["tgt"], max_length=96, truncation=True)["input_ids"]
    return x

train_ds = Dataset.from_list(train).map(tokenize, batched=True, remove_columns=["src", "tgt"])

args = Seq2SeqTrainingArguments(
    output_dir="cleanspeak-lora",
    per_device_train_batch_size=16,
    learning_rate=5e-4,              # LoRA tolerates a much higher LR than full fine-tuning
    num_train_epochs=2,
    logging_steps=50,                # watch the loss: it should go down steadily
    save_strategy="no", report_to="none",
    fp16=False,                      # keep fp32 (T4 has no bf16, and T5 + fp16 is unstable)
    remove_unused_columns=False,
)
trainer = Seq2SeqTrainer(model=model_ft, args=args, train_dataset=train_ds,
                         data_collator=DataCollatorForSeq2Seq(tok_ft, model=model_ft))
trainer.train()
model_ft.eval()

Map:   0%|          | 0/25924 [00:00<?, ? examples/s]

Step,Training Loss
50,0.953441
100,0.811674
150,0.764581
200,0.729613
250,0.676722
300,0.705111
350,0.742243
400,0.732138
450,0.673998
500,0.710003


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32100, 1024)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32100, 1024)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=1024, out_features=1024, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1024, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=1024, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
      

In [36]:
# Save trained LoRA adapter
model_ft.save_pretrained("cleanspeak-lora")

# Save tokenizer
tok_ft.save_pretrained("cleanspeak-lora")

print("Model and tokenizer saved!")

Model and tokenizer saved!


In [35]:
# CELL 8: helpers (generation, bad-word check, meaning score)
from better_profanity import profanity
from sentence_transformers import SentenceTransformer, util

EXTRA_BAD = ["rascal", "bastard", "idiot", "moron", "clown", "stupid", "jerk"]
profanity.load_censor_words()            # built-in list first...
profanity.add_censor_words(EXTRA_BAD)    # ...then ADD ours (load_censor_words(x) would REPLACE the defaults)

def still_dirty(text):
    return [w for w in re.findall(r"[\w']+", text) if profanity.contains_profanity(w)]

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def generate(texts, prefix=PREFIX, bs=32):
    """Run the model (with whatever adapter state is active) on a list of sentences."""
    outs = []
    for i in range(0, len(texts), bs):
        enc = tok_ft([prefix + t for t in texts[i:i + bs]], return_tensors="pt",
                     padding=True, truncation=True, max_length=96).to(DEVICE)
        with torch.no_grad():
            gen = model_ft.generate(**enc, max_length=96, num_beams=4)
        outs += tok_ft.batch_decode(gen, skip_special_tokens=True)
    return outs

def polish(texts):
    """Optional 2nd pass: ORIGINAL CoEdIT (adapter switched off) fixes capitalization/punctuation."""
    with model_ft.disable_adapter():
        return generate(texts, prefix="Fix the grammar: ")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [37]:
# CELL 9: evaluate on the held-out ParaDetox test pairs (never seen in training)
test_src = [p["src"][len(PREFIX):] for p in real_test[:200]]
test_ref = [p["tgt"] for p in real_test[:200]]
outs = generate(test_src)

def avg_meaning(a, b):
    ea, eb = embedder.encode(a, convert_to_tensor=True), embedder.encode(b, convert_to_tensor=True)
    return util.pairwise_cos_sim(ea, eb).mean().item()

print("bad words in INPUT  :", sum(bool(still_dirty(t)) for t in test_src), "/ 200")
print("bad words in OUTPUT :", sum(bool(still_dirty(t)) for t in outs), "/ 200   <- want this near 0")
print("meaning vs input    :", round(avg_meaning(outs, test_src), 3))
print("meaning vs reference:", round(avg_meaning(outs, test_ref), 3))
for s, o, r in list(zip(test_src, outs, test_ref))[:8]:
    print(f"\nIN : {s}\nOUT: {o}\nREF: {r}")

bad words in INPUT  : 101 / 200
bad words in OUTPUT : 14 / 200   <- want this near 0
meaning vs input    : 0.877
meaning vs reference: 0.808

IN : thats not a policy , thats nothing but bull pucky for the ignorant masses .
OUT: That's not a policy, that's nothing for the masses.
REF: Thats not a policy

IN : so i would presume their use of our tax dollars amount to over a million dollars idiot ! .
OUT: so i would presume their use of our tax dollars amount to over a million dollars
REF: so i would presume their use of our tax dollars amount to over a million dollars ! .

IN : you must have some other enemies , now do yourself a favor and fuck off
OUT: you must have some other enemies , now do yourself a favor and leave
REF: You must have some other enemies, now do yourself a favor and go.

IN : as most ( 70 ) of thinking people do not care what these narcissistic morons think .
OUT: as most (70 ) of thinking people do not care what these people think .
REF: as most ( 70 ) of thinking p

In [46]:
# apples-to-apples on the SAME 200 held-out sentences: original CoEdIT (adapter off) vs fine-tuned
with model_ft.disable_adapter():
    base_out = generate(test_src, prefix="Write in a more neutral way: ")

def report(name, o):
    print(f"{name:12} bad words left: {sum(bool(still_dirty(t)) for t in o):3}/200 | "
          f"meaning vs input: {avg_meaning(o, test_src):.3f} | vs ref: {avg_meaning(o, test_ref):.3f}")

report("input", test_src); report("base CoEdIT", base_out); report("fine-tuned", outs)

# look at the 14 leftovers: the list may be flagging mild words, or the model may be missing real ones
for s, o in zip(test_src, outs):
    if still_dirty(o): print(still_dirty(o), "|", s, "->", o)

input        bad words left: 101/200 | meaning vs input: 1.000 | vs ref: 0.768
base CoEdIT  bad words left:  45/200 | meaning vs input: 0.923 | vs ref: 0.788
fine-tuned   bad words left:  14/200 | meaning vs input: 0.877 | vs ref: 0.808
['pot'] | does excessive pot consumption make you stupid ? . -> Does excessive pot consumption make you unintelligent?
['idiot'] | you want us to start seeing motorcycles , dont drive like an idiot . -> you want us to start seeing motorcycles , dont drive like an idiot .
['kill'] | you dont see women lying in wait or going to a workplace to kill their exes . -> you dont see women lying in wait or going to a workplace to kill their exes .
['omg', 'stupid'] | omg , stephens is so silly . -> omg , stephens is so stupid .
['omg', 'stupid'] | omg , stephens is so silly . -> omg , stephens is so stupid .
['gay'] | soon men wont bother with women , they will become gay by choice . -> soon men wont bother with women , they will become gay by choice .
['sucking'

In [38]:
# CELL 10: your 8 baseline sentences (compare against the pipeline results you saved)
TESTS = [
    "bro i is gonna go there, its lit af",
    "you are a bastard and you never listen",
    "that rascal stole my bike yesterday",
    "this is bullshit, the teacher dont know nothing",
    "tbh he's a clown, idk why u trust him",
    "I seen the movie yesterday and it was realy good",
    "I need to assess the class schedule",
    "The meeting starts at 10 AM tomorrow.",
]
ours = generate(TESTS)
polished = polish(ours)
for t, o, p in zip(TESTS, ours, polished):
    print(f"IN      : {t}\nOURS    : {o}\nPOLISHED: {p}\n")

IN      : bro i is gonna go there, its lit af
OURS    : bro i is gonna go there, its lit af
POLISHED: Bro, I am gonna go there. It's lit af

IN      : you are a bastard and you never listen
OURS    : You never listen
POLISHED: You never listen.

IN      : that rascal stole my bike yesterday
OURS    : That guy stole my bike yesterday.
POLISHED: That guy stole my bike yesterday.

IN      : this is bullshit, the teacher dont know nothing
OURS    : this is bad, the teacher dont know nothing
POLISHED: This is bad; the teacher doesn't know nothing.

IN      : tbh he's a clown, idk why u trust him
OURS    : tbh he's not good, idk why u trust him
POLISHED: Tbh, he's not good, I don't know why you trust him.

IN      : I seen the movie yesterday and it was realy good
OURS    : I saw the movie yesterday and it was really good
POLISHED: I saw the movie yesterday, and it was really good.

IN      : I need to assess the class schedule
OURS    : I need to assess the class schedule
POLISHED: I need t

In [39]:
# CELL 11: forgetting check. Same grammar-only sentences, adapter OFF (original CoEdIT) vs adapter ON.
REGRESSION = ["She dont like apples", "He go to school every day", "Their going to the park tomorrow.",
              "The meeting starts at 10 AM tomorrow.", "I need to assess the class schedule."]
with model_ft.disable_adapter():
    before = generate(REGRESSION, prefix="Fix the grammar: ")
after = generate(REGRESSION, prefix="Fix the grammar: ")   # old instruction, fine-tuned weights
for s, b, a in zip(REGRESSION, before, after):
    print(f"IN    : {s}\nBEFORE: {b}\nAFTER : {a}\n")
# if AFTER is clearly worse than BEFORE on these, you have forgetting: lower epochs/LR or add more replay data

IN    : She dont like apples
BEFORE: She doesn't like apples.
AFTER : She doesn't like apples.

IN    : He go to school every day
BEFORE: He goes to school every day.
AFTER : He goes to school every day.

IN    : Their going to the park tomorrow.
BEFORE: They're going to the park tomorrow.
AFTER : They're going to the park tomorrow.

IN    : The meeting starts at 10 AM tomorrow.
BEFORE: The meeting starts at 10 AM tomorrow.
AFTER : The meeting starts at 10 AM tomorrow.

IN    : I need to assess the class schedule.
BEFORE: I need to assess the class schedule.
AFTER : I need to assess the class schedule.



In [47]:
from huggingface_hub import HfApi, notebook_login
notebook_login()    # uncomment only if this errors with an auth message

In [48]:
from huggingface_hub import login
from google.colab import userdata

# Explicit token wins over the auto-detected HF_TOKEN
login(token=userdata.get("HF_WRITE_TOKEN"))

In [41]:
# CELL 12: save the adapter (a few MB) and optionally push it to Hugging Face
model_ft.save_pretrained("coedit-paradetox-lora")
tok_ft.save_pretrained("coedit-paradetox-lora")

('coedit-paradetox-lora/tokenizer_config.json',
 'coedit-paradetox-lora/tokenizer.json')

In [51]:
REPO = "Shinjan123/coedit-paradetox-lora"
model_ft.push_to_hub(REPO, private=True)   # CoEdIT is CC BY-NC 4.0, so mark the card non-commercial
tok_ft.push_to_hub(REPO, private=True)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  42%|####2     | 7.99MB / 18.9MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Shinjan123/coedit-paradetox-lora/commit/1929f80e06e37ac1d244160d43e329e61b422988', commit_message='Upload tokenizer', commit_description='', oid='1929f80e06e37ac1d244160d43e329e61b422988', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Shinjan123/coedit-paradetox-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Shinjan123/coedit-paradetox-lora'), pr_revision=None, pr_num=None)

In [52]:
print(HfApi().list_repo_files(REPO))

['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
